In [ ]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

In [ ]:
from birddog.tracker import (
    PageTracker,
    DynamoDBPageChangeLogTable,
    DynamoDBPageTrackerTable,
    SQLitePageChangeLogTable,
    SQLitePageTrackerTable,
    PageChangeLog,
    )

from birddog.wiki import (
    get_recent_changes,
    lookup_namespace_id,
    )

from birddog.database import Database
from birddog.store import KeyValueStore, DynamoDBKeyValueStore
from birddog.utility import json_size, now

In [ ]:
#def copy_page_tracker_to_ddb(batch_size=100, limit=None):
#    ddb_table = DynamoDBPageTrackerTable()
#    page_tracker = PageTracker()
#    entries = list(page_tracker._page_dict.items())
#    if not limit:
#        limit = len(entries)
#    print(f"pushing {limit} entries to DDB, batch_size={batch_size}")
#    for i in range(0, limit, batch_size):
#        print(f"batch {i}")
#        batch = { title: update for title, update in entries[i:(i+batch_size)] }
#        ddb_table.put(batch)

In [ ]:
#copy_page_tracker_to_ddb(batch_size=1000)

In [ ]:
#tracker = PageTracker()

In [ ]:
#changes = PageChangeLog()

In [ ]:
wikisource_file_ns = "Файл"
commons_file_ns = "File"
commons_base = "https://commons.wikimedia.org"

In [ ]:
lookup_namespace_id(wikisource_file_ns)

In [ ]:
lookup_namespace_id(commons_file_ns)

In [ ]:
c=get_recent_changes(cutoff_date="2026,02,03,23:00", base=commons_base, namespace=6)

In [ ]:
len(c)

In [ ]:
now(universal=True)

In [ ]:
list(c.items())[:1]

In [ ]:
max([v["timestamp"] for v in c.values()])

In [ ]:
min([v["timestamp"] for v in c.values()])

In [ ]:
len(c)/5

In [ ]:
db = Database()

In [ ]:
dids = db.get_all_ids("Documents")

In [ ]:
doc_recs = db.read("Documents", dids)

In [ ]:
commons_prefix = "https://commons.wikimedia.org/wiki/File:"

In [ ]:
[d["title"] for d in doc_recs[:10] if d.get("link", "").startswith(commons_prefix)]

In [ ]:
commons_titles = { d["title"]: {"Id": d.get("Id"), "timestamp": d.get("timestamp")} 
                   for d in doc_recs 
                   if d.get("link", "").startswith(commons_prefix)
                 }

In [ ]:
#commons_titles

In [ ]:
commons_changes = [
    change for change in c 
    if change[0] in commons_titles
    ]

In [ ]:
commons_changes

In [ ]:
doc_store = KeyValueStore(table_name="doc_titles")
doc_ns = "commons"

In [ ]:
list(commons_titles.items())[:1]

In [ ]:
len(commons_titles)

In [ ]:
def store_titles(store, titles, ns=doc_ns):
    for title, entry in titles.items():
        store.insert(ns, title, str(entry.get("Id", "")))

In [ ]:
store_titles(doc_store, commons_titles)

In [ ]:
t = doc_store.get_all(doc_ns)

In [ ]:
len(t)

In [ ]:
t[:10]

In [ ]:
json_size(commons_titles)